In [1]:
!pip uninstall scikit-surprise numpy -y
# usar numpy menor que 2 para que scikit-surprise (compilado con numpy 1.x) funcione
!pip install "numpy<2"
!pip install scikit-surprise

Found existing installation: scikit-surprise 1.1.4
Uninstalling scikit-surprise-1.1.4:
  Successfully uninstalled scikit-surprise-1.1.4
Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Defaulting to user installation because normal site-packages is not writeable
  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.

[notice] A new release of pip is available: 23.1.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable
  Using ca

In [5]:
import pandas as pd
from surprise import Reader, Dataset
from surprise.model_selection import train_test_split
from surprise import SVD
from surprise import KNNBasic
from surprise import accuracy
from sklearn.preprocessing import MinMaxScaler, LabelEncoder

In [16]:
df=pd.read_csv('https://breathecode.herokuapp.com/asset/internal-link?id=2326&path=adult-census-income.csv')
df

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,22,Private,310152,Some-college,10,Never-married,Protective-serv,Not-in-family,White,Male,0,0,40,United-States,<=50K
32557,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States,<=50K
32558,40,Private,154374,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,>50K
32559,58,Private,151910,HS-grad,9,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,<=50K


In [ ]:
# 1. Limpieza: Eliminamos nulos y columnas irrelevantes
df = df.dropna()
# renombrar columnas para que coincidan con nombres esperados
df = df.rename(columns={
    'education.num':'education-num',
    'capital.gain':'capital-gain',
    'capital.loss':'capital-loss',
    'hours.per.week':'hours-per-week',
    'marital.status':'marital-status',
    'native.country':'native-country'
})
df = df.drop(columns=['fnlwgt', 'education']) # Usamos education-num que ya es numérica

# 2. Transformamos la variable objetivo (Income) a numérica
# >50K será 1, <=50K será 0
df['income'] = df['income'].apply(lambda x: 1 if '>' in str(x) else 0)

# 3. Normalizar variables numéricas (Age, Hours-per-week, etc.)
scaler = MinMaxScaler()
num_cols = ['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
df[num_cols] = scaler.fit_transform(df[num_cols])

# 4. Codificación de categóricas para el perfil de usuario
# Guardamos los encoders para usarlos después en las pruebas
le_dict = {}
cat_cols = ['workclass', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    le_dict[col] = le


# Definición del Problema de Recomendación

¿Qué se quiere recomendar?
- combinaciones de Ocupación y Nivel Educativo que maximicen el ingreso.

¿Cuál será el "usuario"? 
- Un individuo definido por sus características demográficas (Edad, Sexo, Raza, Estado Civil, etc.).

¿Qué variables definen el perfil? 
- Todas las demográficas que no estamos tratando como "recomendación".

El "Rating": En Surprise necesitamos un puntaje. 
- Usaremos la probabilidad de ganar >50K.

In [19]:
from surprise import Reader, Dataset, SVD
from surprise.model_selection import train_test_split

# Creamos una columna 'item_id' que sea la combinación de ocupación y educación
df['item_id'] = df['occupation'].astype(str) + "_" + df['education.num'].astype(str)

# El "user_id" será simplemente el índice del dataframe para este ejercicio
df['user_id'] = df.index

# Surprise necesita: user, item, rating
# Usamos 'income' como el rating (0 o 1)
reader = Reader(rating_scale=(0, 1))
data = Dataset.load_from_df(df[['user_id', 'item_id', 'income']], reader)

# Entrenar el modelo
trainset, testset = train_test_split(data, test_size=0.2)
model = SVD()
model.fit(trainset)

print("✅ Modelo SVD entrenado con éxito.")

✅ Modelo SVD entrenado con éxito.


In [20]:
# Simulemos un usuario: Joven, educación media, soltero
user_id_simulado = 999999 

# Lista de todas las posibles combinaciones de (Ocupación_Educación) únicas en el dataset
todas_las_trayectorias = df['item_id'].unique()

predictions = []
for trayec in todas_las_trayectorias:
    pred = model.predict(user_id_simulado, trayec)
    predictions.append((trayec, pred.est))

# Ordenar por las mejores recomendaciones (mayor probabilidad de >50K)
predictions.sort(key=lambda x: x[1], reverse=True)

print(f"Top 3 recomendaciones para el usuario {user_id_simulado}:")
for i in range(3):
    trayec_id, score = predictions[i]
    occ_code = int(trayec_id.split('_')[0])
    # Revertir el nombre de la ocupación usando nuestro diccionario de encoders
    occ_name = le_dict['occupation'].inverse_transform([occ_code])[0]
    print(f"{i+1}. {occ_name} (Score estimado: {score:.2f})")

Top 3 recomendaciones para el usuario 999999:
1. Exec-managerial (Score estimado: 0.83)
2. Prof-specialty (Score estimado: 0.77)
3. Exec-managerial (Score estimado: 0.76)


# Cosideraciones finales
Limpiamos los ? y convertimos el ingreso en 0 y 1.

Estructuramos el problema: el "Rating" es la probabilidad de éxito económico.

SVD encontró patrones: por ejemplo, descubrió que ciertas ocupaciones combinadas con ciertos niveles educativos "puntúan mejor" para perfiles similares al del usuario simulado.

Recomendamos: le dijimos al usuario en qué ocupación tendría más probabilidad de superar los 50K anuales.